# Dataset Analysis

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Imports

In [ ]:
# --------------------------- Standard Libraries ---------------------------- #
import os
import seaborn as sns
from IPython.display import display

# -------------------------------- Annotations ------------------------------- #
from typing import List, Optional, Tuple

# ------------------------- Data Processing Libraries ----------------------- #
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import fireducks.pandas as pd

## 2. Utility Functions Definitions

### 2.1. Folder and Files

In [ ]:
def create_run_directory(prefix: str, base_dir: str = "runs") -> str:
    """
    Creates a new directory for storing training logs, checkpoints, and plots.
    The directory name is based on the next available number.

    Args:
        prefix (str): Prefix for the run directory.
        base_dir (str): Base directory for storing training runs. Defaults to "runs".

    Returns:
        str: Path to the created run directory.
    """
    os.makedirs(base_dir, exist_ok=True)  # Ensure the base directory exists

    # Find the next available run number
    existing_dirs = [d for d in os.listdir(base_dir) if d.startswith(prefix) and d[len(prefix):].isdigit()]
    next_run_number = max([int(d[len(prefix):]) for d in existing_dirs] + [0]) + 1
    run_dir = os.path.join(base_dir, f"{prefix}{next_run_number}")
    os.makedirs(run_dir, exist_ok=True)  # Create the run directory

    return run_dir

### 2.2. Plotting

In [ ]:
def plot_scientific(
    x: List[float],
    y_datasets: List[List[float]],
    labels: Optional[List[str]] = None,
    x_label: str = "X-axis",
    y_label: str = "Y-axis",
    title: str = "Scientific Plot",
    x_integer: bool = False,
    y_log: bool = False,
    markers: Optional[List[str]] = None,
    line_styles: Optional[List[str]] = None,
    legend_loc: str = "best",
    grid: bool = True,
    xlim: Optional[Tuple[float, float]] = None,
    ylim: Optional[Tuple[float, float]] = None,
    save_path: Optional[str] = None,
    dpi: int = 300,
) -> None:
    """
    Plots multiple Y datasets against a shared X-axis with scientific paper styling.

    Args:
        x (List[float]): List of X-axis values.
        y_datasets (List[List[float]]): List of lists containing Y-axis datasets.
        labels (Optional[List[str]]): Labels for each dataset for the legend.
        x_label (str): Label for the X-axis. Default is "X-axis".
        y_label (str): Label for the Y-axis. Default is "Y-axis".
        title (str): Title of the plot. Default is "Scientific Plot".
        x_integer (bool): Force X-axis to display only integer values. Default is False.
        y_log (bool): Use logarithmic scale for the Y-axis. Default is False.
        markers (Optional[List[str]]): List of marker styles for each dataset.
        line_styles (Optional[List[str]]): List of line styles for each dataset.
        legend_loc (str): Location of the legend. Default is "best".
        grid (bool): Whether to display a grid. Default is True.
        xlim (Optional[Tuple[float, float]]): Limits for the X-axis as (min, max). Default is None.
        ylim (Optional[Tuple[float, float]]): Limits for the Y-axis as (min, max). Default is None.
        save_path (Optional[str]): Path to save the plot as a file. Default is None.
        dpi (int): Resolution of the saved plot in dots per inch. Default is 300.

    Returns:
        None: Displays the plot and optionally saves it as an image.
    """
    # Validate input dimensions
    if any(len(y) != len(x) for y in y_datasets):
        raise ValueError("All Y datasets must have the same length as the X dataset.")

    # Initialize the plot
    plt.figure(figsize=(8, 6))

    # Plot each dataset
    for i, y in enumerate(y_datasets):
        label = labels[i] if labels and i < len(labels) else f"Dataset {i + 1}"
        marker = markers[i] if markers and i < len(markers) else "o"
        line_style = line_styles[i] if line_styles and i < len(line_styles) else "-"
        plt.plot(x, y, label=label, marker=marker, linestyle=line_style)

    # Configure axes and title
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel(y_label, fontsize=12)
    plt.title(title, fontsize=14, weight="bold")
    plt.legend(loc=legend_loc, fontsize=10)

    # Configure axis limits
    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)

    # Configure X-axis for integers only
    if x_integer:
        plt.xticks(ticks=range(int(min(x)), int(max(x)) + 1))

    # Enable logarithmic scale for Y-axis if requested
    if y_log:
        plt.yscale("log")

    # Add grid if requested
    if grid:
        plt.grid(visible=True, linestyle="--", linewidth=0.5, alpha=0.7)

    # Final styling
    plt.tight_layout()

    # Save plot if path is provided
    if save_path:
        plt.savefig(save_path, dpi=dpi, format="png")

    # Show plot
    plt.show()
    

def plot_histogram(
    datasets: List[List[float]],
    bins: int = 10,
    density: bool = False,
    labels: Optional[List[str]] = None,
    colors: Optional[List[str]] = None,
    edgecolors: Optional[List[str]] = None,
    alpha: float = 0.7,
    x_label: str = "X-axis",
    y_label: str = "Frequency",
    title: str = "Histogram",
    legend_loc: str = "best",
    grid: bool = True,
    xlim: Optional[Tuple[float, float]] = None,
    ylim: Optional[Tuple[float, float]] = None,
    save_path: Optional[str] = None,
    dpi: int = 300,
) -> None:
    """
    Plots a histogram for one or more datasets with scientific styling.

    Args:
        datasets (List[List[float]]): List of datasets to plot histograms for.
        bins (int): Number of bins in the histogram. Default is 10.
        density (bool): If True, normalizes the histogram so the area equals 1. Default is False.
        labels (Optional[List[str]]): Labels for each dataset for the legend. Default is None.
        colors (Optional[List[str]]): Colors for each dataset. Default is None.
        edgecolors (Optional[List[str]]): Edge colors for each dataset. Default is None.
        alpha (float): Transparency level of bars (0: fully transparent, 1: opaque). Default is 0.7.
        x_label (str): Label for the X-axis. Default is "X-axis".
        y_label (str): Label for the Y-axis. Default is "Frequency".
        title (str): Title of the histogram. Default is "Histogram".
        legend_loc (str): Location of the legend. Default is "best".
        grid (bool): Whether to display a grid. Default is True.
        xlim (Optional[Tuple[float, float]]): Limits for the X-axis as (min, max). Default is None.
        ylim (Optional[Tuple[float, float]]): Limits for the Y-axis as (min, max). Default is None.
        save_path (Optional[str]): Path to save the histogram as a file. Default is None.
        dpi (int): Resolution of the saved histogram in dots per inch. Default is 300.

    Returns:
        None: Displays the histogram and optionally saves it as an image.
    """
    # Initialize the plot
    plt.figure(figsize=(8, 6))

    # Plot each dataset as a histogram
    for i, data in enumerate(datasets):
        label = labels[i] if labels and i < len(labels) else f"Dataset {i + 1}"
        color = colors[i] if colors and i < len(colors) else None
        edgecolor = edgecolors[i] if edgecolors and i < len(edgecolors) else None
        plt.hist(
            data,
            bins=bins,
            density=density,
            alpha=alpha,
            label=label,
            color=color,
            edgecolor=edgecolor,
        )

    # Configure axes and title
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel(y_label if not density else "Density", fontsize=12)
    plt.title(title, fontsize=14, weight="bold")

    # Configure axis limits
    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)

    # Add grid if requested
    if grid:
        plt.grid(visible=True, linestyle="--", linewidth=0.5, alpha=0.7)

    # Add legend if labels are provided
    if labels:
        plt.legend(loc=legend_loc, fontsize=10)

    # Final styling
    plt.tight_layout()

    # Save histogram if path is provided
    if save_path:
        plt.savefig(save_path, dpi=dpi, format="png")

    # Show histogram
    plt.show()
    
    
def plot_boxplot_scientific(
    dataset: np.ndarray,
    x_label: str = "Columns",
    y_label: str = "Values",
    title: str = "Boxplot of Dataset Columns",
    figsize: Tuple[int, int] = (12, 8),
    box_color: str = "gray",
    flier_color: str = "black",
    flier_size: int = 6,
    grid: bool = True,
    grid_style: str = "--",
    grid_alpha: float = 0.7,
    xlim: Optional[Tuple[float, float]] = None,
    ylim: Optional[Tuple[float, float]] = None,
    tick_rotation: int = 0,
    dpi: int = 300,
    save_path: Optional[str] = None,
    hide_xlabels: bool = False,
) -> Optional[str]:
    """
    Generates a highly configurable scientific-style boxplot.

    Args:
        dataset (np.ndarray): The dataset as a 2D NumPy array.
        x_label (str): Label for the X-axis. Default is "Columns".
        y_label (str): Label for the Y-axis. Default is "Values".
        title (str): Title of the plot. Default is "Boxplot of Dataset Columns".
        figsize (Tuple[int, int]): Size of the figure in inches. Default is (12, 8).
        box_color (str): Color of the boxes in the boxplot. Default is "gray".
        flier_color (str): Color of outlier points. Default is "black".
        flier_size (int): Size of the outlier points. Default is 6.
        grid (bool): Whether to show grid lines. Default is True.
        grid_style (str): Style of the grid lines. Default is "--".
        grid_alpha (float): Transparency of the grid lines. Default is 0.7.
        xlim (Optional[Tuple[float, float]]): Limits for the X-axis. Default is None.
        ylim (Optional[Tuple[float, float]]): Limits for the Y-axis. Default is None.
        tick_rotation (int): Rotation angle for X-axis tick labels. Default is 45.
        dpi (int): DPI for the saved plot image. Default is 300.
        save_path (Optional[str]): File path to save the plot. If None, the plot is displayed.
        hide_xlabels (bool): Whether to hide X-axis labels. Default is False.
    
    Returns:
        Optional[str]: The file path of the saved plot, or None if not saved.
    """
    # Create the plot
    plt.figure(figsize=figsize)
    sns.boxplot(
        data=dataset,
        orient="v",
        color=box_color,
        fliersize=flier_size,
        linewidth=1.5,
        flierprops=dict(marker='o', color=flier_color, markerfacecolor=flier_color),
    )

    # Configure titles and labels
    plt.title(title, fontsize=18, weight="bold")
    plt.xlabel(x_label, fontsize=16)
    plt.ylabel(y_label, fontsize=16)

    # Configure axis limits
    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)

    # Configure ticks
    plt.xticks(fontsize=12, rotation=tick_rotation)
    plt.yticks(fontsize=12)

    # Hide X-axis labels if requested
    if hide_xlabels:
        plt.xticks([])

    # Configure grid
    if grid:
        plt.grid(visible=True, linestyle=grid_style, linewidth=0.5, alpha=grid_alpha)

    # Save or show the plot
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")   
    plt.show()

## 3. Setup and Configuration

### 3.1. Folder and File Management

In [ ]:
# Create the base directory for the current run:
RUN_DIR = create_run_directory(prefix="dataset_analysis_")
FIG_DIR = os.path.join(RUN_DIR, "figures")

os.makedirs(FIG_DIR, exist_ok=True)


## 4. Data Loading and Preprocessing

### 5.1. Data Loading

In [ ]:
# --------------------- Load the dataset in matlab format -------------------- #
#! WARNING: the input data should be a column vector
dataset = pd.DataFrame((scipy.io.loadmat('./data/shadowed_rayleigh/SNR_events.mat')['SNR_events']))

print(f"Original dataset shape: {dataset.shape}")

# ---------------------------- Data Configuration ---------------------------- #
# Subsample data for testing, drop some columns
dataset = dataset.sample(frac=0.01, axis=1)  # Take x% of the columns

# ------------------------------ Scale the data ------------------------------ #
#! Warning: this makes the numbers much closer, which may make training harder
dataset = np.log10(dataset)

print(f"Shape of the data after configuration: {dataset.shape}\n")

### 5.2. Data Analytics

In [ ]:
print("-----------------------------------------------")
print("Concise summary:")
dataset.info()
print("-----------------------------------------------")

print("\nFirst rows:")
display(dataset.head(n=10))

print("Last rows:")
display(dataset.tail(n=10))

In [ ]:
#! WARNING: this takes up a lot of time and memory
print("\nSummary statistics for numerical columns:")
summary = dataset.describe()
display(summary)

print("\nMean of Summary statistics rows:")
display(summary.loc[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].mean(axis=1))

In [ ]:
#! WARNING: this takes up a lot of time and memory
# print("Correlation matrix:")
# display(dataset.corr())

# print("\nCovariance between columns:")
# display(dataset.cov())

In [ ]:
# Count total NaN and Inf values
total_nan, total_inf = dataset.isna().sum().sum(), np.isinf(dataset.values).sum()

# Identify columns containing NaN or Inf
nan_cols, inf_cols = dataset.isna().sum(), np.isinf(dataset).sum()
nan_cols, inf_cols = nan_cols[nan_cols > 0], inf_cols[inf_cols > 0]

# Check for duplicate columns
duplicate_cols = dataset.T.duplicated().sum()

print(f"\nTotal NaN values: {total_nan}\nTotal Inf values: {total_inf}")
print("\nDuplicate Columns Count:", duplicate_cols)

# print("\nColumns with NaN values:\n", nan_cols if not nan_cols.empty else "No NaN values detected.")
# print("\nColumns with Inf values:\n", inf_cols if not inf_cols.empty else "No Inf values detected.")

# duplicate_cols_list = dataset.loc[:, dataset.T.duplicated()].columns
# print("\nDuplicate Columns:\n", duplicate_cols_list.tolist() if not duplicate_cols_list.empty else "No duplicate columns found.")


Total NaN values: 0
Total Inf values: 0

Duplicate Columns Count: 0

Columns with NaN values:
 No NaN values detected.

Columns with Inf values:
 No Inf values detected.

Duplicate Columns:
 No duplicate columns found.


In [ ]:
# ---------------------------- Plot the histogram ---------------------------- #
plot_histogram(
    datasets=[dataset],
    bins=100,
    edgecolors=["black"],
    title="Histogram of Data",
    x_label="SNR",
    y_label="Frequency",
    save_path=os.path.join(FIG_DIR, "histogram.png"),
)

# ----------------------------- Plot the Boxplot ----------------------------- #
# Sample x random columns for boxplot
sample_columns = np.random.choice(
    dataset.shape[1],
    size=72,  # X Random columns
    replace=False,
)


plot_boxplot_scientific(
    dataset=dataset[:, sample_columns],
    x_label="",
    y_label="Values",
    title="Boxplot of Sampled Columns",
    figsize=(20, 12),
    box_color="black",
    flier_color="black",
    hide_xlabels=True,
    save_path=os.path.join(FIG_DIR, "boxplot_sampled_columns_scientific.png"),
)